# Build Datasets

In [1]:
pip install transformers datasets torch scikit-learn pandas tqdm

ERROR: datasets 3.1.0 has requirement requests>=2.32.2, but you'll have requests 2.24.0 which is incompatible.
ERROR: datasets 3.1.0 has requirement tqdm>=4.66.3, but you'll have tqdm 4.47.0 which is incompatible.



  Using cached aiosignal-1.3.1-py3-none-any.whl (7.6 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 0.7.4
    Uninstalling fsspec-0.7.4:
      Successfully uninstalled fsspec-0.7.4


# label의 의미
## 0 : negative
## 1 : neutral/none
## 2 : positive

In [2]:
import json
import pandas as pd
from tqdm import tqdm


# Aspect 정의
ASPECTS = ["맛", "서비스", "분위기", "양", "가격"]

# label
# 0 = negative
# 1 = neutral
# 2 = positive


# Positive 태그 매핑
POSITIVE_TAG_MAP = {
    "음식이 맛있어요": ("맛", 2),
    "재료가 신선해요": ("맛", 2),
    "고기 질이 좋아요": ("맛", 2),

    "친절해요": ("서비스", 2),
    "음식이 빨리 나와요": ("서비스", 2),

    "인테리어가 멋져요": ("분위기", 2),
    "매장이 청결해요": ("분위기", 2),
    "아늑해요": ("분위기", 2),
    "대화하기 좋아요": ("분위기", 2),

    "양이 많아요": ("양", 2),

    "가성비가 좋아요": ("가격", 2),
}


# Negative 패턴 -> 다 같이 모여서 수정하는 것도 좋을듯!!
NEGATIVE_PATTERNS = {
    "맛": [
        "맛없", "별로", "짜다", "싱겁",
        "느끼", "비리", "차갑", "탄맛"
    ],

    "서비스": [
        "불친절", "싸가지", "느리",
        "대응 별로", "서비스 별로"
    ],

    "분위기": [
        "시끄럽", "더럽", "좁",
        "불편", "냄새"
    ],

    "양": [
        "양 적", "적다", "부족"
    ],

    "가격": [
        "비싸", "창렬", "가성비 별로"
    ]
}


# JSON 로드
with open("naver_reviews.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = []


# 데이터 생성
for restaurant_id, restaurant_data in tqdm(data.items()):

    reviews = restaurant_data["reviews"]

    for review in reviews:

        text = review.get("content", "")
        tags = review.get("tags", [])

        # 기본값 neutral
        aspect_labels = {
            aspect: 1 for aspect in ASPECTS
        }


        # Positive 태그 적용
        for tag in tags:

            if tag in POSITIVE_TAG_MAP:
                aspect, label = POSITIVE_TAG_MAP[tag]
                aspect_labels[aspect] = label


        # Negative 패턴 적용
        for aspect, patterns in NEGATIVE_PATTERNS.items():

            for pattern in patterns:

                if pattern in text:
                    aspect_labels[aspect] = 0


        # Aspect-aware sample 생성
        for aspect in ASPECTS:

            dataset.append({
                "text": f"[{aspect}] {text}",
                "aspect": aspect,
                "label": aspect_labels[aspect]
            })


# DataFrame 저장
df = pd.DataFrame(dataset)

print(df.head())

print("\nLabel Distribution")
print(df["label"].value_counts())

df.to_csv(
    "aspect_sentiment_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved!")

100%|██████████████████████████████████████████████████████████████████████████████| 825/825 [00:00<00:00, 2012.36it/s]


                                                text aspect  label
0  [맛] 항상 돈까스하면 생각나는 곳이네요 ㅎㅎ 돈까스에 라면 너무 맛있습니다 ㅎㅎ ...      맛      2
1  [서비스] 항상 돈까스하면 생각나는 곳이네요 ㅎㅎ 돈까스에 라면 너무 맛있습니다 ㅎ...    서비스      1
2  [분위기] 항상 돈까스하면 생각나는 곳이네요 ㅎㅎ 돈까스에 라면 너무 맛있습니다 ㅎ...    분위기      1
3  [양] 항상 돈까스하면 생각나는 곳이네요 ㅎㅎ 돈까스에 라면 너무 맛있습니다 ㅎㅎ ...      양      2
4  [가격] 항상 돈까스하면 생각나는 곳이네요 ㅎㅎ 돈까스에 라면 너무 맛있습니다 ㅎㅎ...     가격      1

Label Distribution
1    114655
2     68709
0      2261
Name: label, dtype: int64

Saved!


# KoELECTRA Fine-tuning

In [1]:
pip install -U accelerate transformers

Requirement already up-to-date: accelerate in c:\python\lib\site-packages (1.0.1)
Requirement already up-to-date: transformers in c:\python\lib\site-packages (4.46.3)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 2070 SUPER


In [3]:
import pandas as pd
import numpy as np

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


# 모델
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"


# 데이터 로드
df = pd.read_csv("aspect_sentiment_dataset.csv")


print(df["label"].value_counts())



# ~ 모델 학습 파트 ~
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)


# 모델
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)


# 평가 함수
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return {
        "accuracy": acc,
        "macro_f1": f1
    }


# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    fp16=True,

    report_to="none",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True
)


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


# 학습 시작
trainer.train()


# 모델 저장
trainer.save_model("./koelectra_aspect_model")
tokenizer.save_pretrained("./koelectra_aspect_model")

print("\nTraining Finished!")

1    114655
2     68709
0      2261
Name: label, dtype: int64


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.525300,0.506615,0.770559,0.786992
2,0.404300,0.500771,0.776350,0.797873



Training Finished!


# Sentiment Analysis

In [5]:
import json
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# 모델 경로
MODEL_PATH = "./koelectra_aspect_model"


# Aspect 정의
ASPECTS = ["맛", "서비스", "분위기", "양", "가격"]


# Label Mapping
LABEL_MAP = {
    0: -1,   # negative
    1: 0,    # neutral
    2: 1     # positive
}

LABEL_NAME = {
    0: "negative",
    1: "neutral",
    2: "positive"
}


# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# 모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)

model.eval()


# JSON 로드
with open("naver_reviews.json", "r", encoding="utf-8") as f:
    data = json.load(f)


# 결과 저장
restaurant_results = {}


# 음식점 반복
for restaurant_id, restaurant_data in tqdm(data.items()):

    restaurant_name = restaurant_data.get(
        "name",
        restaurant_id
    )

    reviews = restaurant_data.get("reviews", [])

    # 음식점별 점수 저장
    aspect_scores = defaultdict(list)

    # 리뷰 분석 결과 저장
    analyzed_reviews = []

    # 리뷰 반복
    for review in reviews:

        review_text = review.get("content", "")

        if len(review_text.strip()) == 0:
            continue

        review_result = {}


        # Aspect별 분석
        for aspect in ASPECTS:

            input_text = f"[{aspect}] {review_text}"

            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )

            inputs = {
                k: v.to(device)
                for k, v in inputs.items()
            }

            with torch.no_grad():

                outputs = model(**inputs)

                probs = torch.softmax(
                    outputs.logits,
                    dim=1
                )

                pred = torch.argmax(
                    probs,
                    dim=1
                ).item()

                confidence = probs[0][pred].item()

            sentiment_score = LABEL_MAP[pred]

            review_result[aspect] = {
                "label": LABEL_NAME[pred],
                "score": sentiment_score,
                "confidence": round(confidence, 4)
            }

            aspect_scores[aspect].append(
                sentiment_score
            )

        analyzed_reviews.append({
            "review": review_text,
            "analysis": review_result
        })


    # 음식점 평균 계산
    final_scores = {}

    for aspect in ASPECTS:

        scores = aspect_scores[aspect]

        if len(scores) == 0:
            avg_score = 0

        else:
            avg_score = np.mean(scores)

        final_scores[aspect] = round(
            float(avg_score),
            3
        )


    # 최종 추천 점수 -> 다 같이 모여서 가중치 조정하는게 좋을듯!!
    recommendation_score = (
        final_scores["맛"] * 0.4 +
        final_scores["서비스"] * 0.2 +
        final_scores["분위기"] * 0.15 +
        final_scores["양"] * 0.15 +
        final_scores["가격"] * 0.1
    )

    recommendation_score = round(
        recommendation_score,
        3
    )


    # 저장
    restaurant_results[restaurant_id] = {

        "restaurant_name": restaurant_name,

        "review_count": len(analyzed_reviews),

        "aspect_scores": final_scores,

        "recommendation_score": recommendation_score,

        "reviews": analyzed_reviews
    }


# 추천 순 정렬
sorted_results = dict(
    sorted(
        restaurant_results.items(),
        key=lambda x: x[1]["recommendation_score"],
        reverse=True
    )
)


# 저장
with open(
    "restaurant_analysis.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        sorted_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nSaved: restaurant_analysis.json")


# TOP 10 출력
print("\n===== TOP 10 Restaurants =====\n")

top10 = list(sorted_results.items())[:10]

for idx, (restaurant_id, info) in enumerate(top10):

    print(f"{idx+1}. {info['restaurant_name']}")

    print(f"추천점수: {info['recommendation_score']}")

    print("카테고리 점수:")

    for aspect, score in info["aspect_scores"].items():

        print(f"  {aspect}: {score}")

    print()

Device: cuda


100%|████████████████████████████████████████████████████████████████████████████████| 825/825 [33:33<00:00,  2.44s/it]



Saved: restaurant_analysis.json

===== TOP 10 Restaurants =====

1. 2047540576
추천점수: 0.643
카테고리 점수:
  맛: 1.0
  서비스: 0.4
  분위기: 0.133
  양: 0.6
  가격: 0.533

2. 1765098518
추천점수: 0.636
카테고리 점수:
  맛: 0.956
  서비스: 0.711
  분위기: 0.289
  양: 0.422
  가격: 0.044

3. 1589373477
추천점수: 0.634
카테고리 점수:
  맛: 0.956
  서비스: 0.622
  분위기: 0.822
  양: 0.0
  가격: 0.044

4. 1504602021
추천점수: 0.63
카테고리 점수:
  맛: 0.978
  서비스: 0.4
  분위기: 0.733
  양: 0.044
  가격: 0.422

5. 2052668536
추천점수: 0.624
카테고리 점수:
  맛: 1.0
  서비스: 0.511
  분위기: 0.756
  양: 0.044
  가격: 0.022

6. 1035431851
추천점수: 0.62
카테고리 점수:
  맛: 0.956
  서비스: 0.533
  분위기: 0.822
  양: 0.022
  가격: 0.044

7. 1962212196
추천점수: 0.619
카테고리 점수:
  맛: 0.933
  서비스: 0.511
  분위기: 0.956
  양: 0.0
  가격: 0.0

8. 1301688032
추천점수: 0.618
카테고리 점수:
  맛: 1.0
  서비스: 0.489
  분위기: 0.8
  양: 0.0
  가격: 0.0

9. 1773802354
추천점수: 0.613
카테고리 점수:
  맛: 1.0
  서비스: 0.444
  분위기: 0.578
  양: 0.089
  가격: 0.244

10. 1783606657
추천점수: 0.612
카테고리 점수:
  맛: 0.956
  서비스: 0.778
  분위기: 0.2
  양: 0.089
  가격: 0.311

